# Homework: Vector Search
- The homework is available at [DataTalksClub Github](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/02-vector-search/homework.md)


---
## Q1. Embedding a query

In [1]:
from embedder import Embedder

embedder = Embedder()

query_text  = "How does approximate nearest neighbor search work?"
query_vector_q1 = embedder.encode(query_text)

first_value = round(query_vector_q1[0],2)
print(f'Q1 — The first value v[0] is {first_value}')

Q1 — The first value v[0] is -0.02


---
# Q2. Cosine similarity

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [3]:
mdfile = next(
    doc for doc in documents 
    if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md")

mdfile_vector = embedder.encode(mdfile['content'])
cosine_similarity = query_vector_q1.dot(mdfile_vector)

In [4]:
print(f'Q2 — The cosine similarity with the query vector from Q1 is {round(cosine_similarity,2)}')

Q2 — The cosine similarity with the query vector from Q1 is 0.36


---
## Q3. Chunking and search by hand

In [5]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [6]:
from tqdm.auto import tqdm
import numpy as np

texts = [chunk['content'] for chunk in chunks]

batch_size = 50
X = []

for i in range(0, len(texts), batch_size):
    batch = texts[i:i + batch_size]
    batch_vectors = embedder.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

In [7]:
scores = X.dot(query_vector_q1)
idx = np.argmax(scores)

filename_q3 = chunks[idx]['filename']
score_q3 = round(scores[idx],2)
print(f'Q3 — The  highest-score chunk\'s filename is "{filename_q3}" with a score: {score_q3}.')

Q3 — The  highest-score chunk's filename is "02-vector-search/lessons/07-sqlitesearch-vector.md" with a score: 0.65.


---
## Q4. Vector search with minsearch

In [8]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

In [9]:
query_q4 = "What metric do we use to evaluate a search engine?"
query_vector_q2 = embedder.encode(query_q4)

result_q4 = vindex.search(query_vector_q2, num_results=1)[0]
filename_q4 = result_q4["filename"]

In [10]:
print(f'Q4 — The  highest-score chunk\'s filename is "{filename_q4}".')

Q4 — The  highest-score chunk's filename is "04-evaluation/lessons/05-search-metrics.md".


---
## Q5. Text search vs vector search

In [11]:
from minsearch import Index

index = Index(text_fields=['content'])
index.fit(chunks)

In [12]:
query_q5 = 'How do I store vectors in PostgreSQL?'

query_vector_q5 = embedder.encode(query_q5)

results_text = index.search(query_q5, num_results=5)
results_vector = vindex.search(query_vector_q5, num_results=5)

In [13]:
text_files = {doc["filename"] for doc in results_text}
vector_files = {doc["filename"] for doc in results_vector}

difference = vector_files - text_files

if difference:
    filename_q5 = next(iter(difference))
    print(f"Q5 — The file '{filename_q5}' appears in vector search but not in text search.")


Q5 — The file '02-vector-search/lessons/08-pgvector.md' appears in vector search but not in text search.


---
## Q6. Hybrid search

In [14]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [15]:
query_q6 = 'How do I give the model access to tools?'
query_vector_q6 = embedder.encode(query_q6)

results_text_q6 = index.search(query_q6, num_results=5)
results_vector_q6 = vindex.search(query_vector_q6, num_results=5)

results_rrf = rrf([results_vector_q6, results_text_q6])

In [16]:
first_rrf = results_rrf[0]["filename"]
print(f'Q6 — After RRF \'{first_rrf}\' is ranked first.')

Q6 — After RRF '01-agentic-rag/lessons/13-function-calling.md' is ranked first.
